In [1]:
import os
import numpy as np
import yt
import trident

yt.funcs.mylog.setLevel("INFO")  # or "WARNING"

In [3]:
import os

CUTOUT = "/scratch/tsingh65/TNG50-1_snap99/out_sub_488530/cutout_ALLFIELDS_sphere_2p1Rvir_sub488530.hdf5"
assert os.path.exists(CUTOUT), f"Missing: {CUTOUT}"
print("CUTOUT:", CUTOUT)
print("Size (GB):", os.path.getsize(CUTOUT)/1e9)

CUTOUT: /scratch/tsingh65/TNG50-1_snap99/out_sub_488530/cutout_ALLFIELDS_sphere_2p1Rvir_sub488530.hdf5
Size (GB): 5.536269336


In [4]:
# Cell 2 — load and basic sanity
import yt
ds = yt.load(CUTOUT)

print("ds:", type(ds))
print("basename:", ds.basename)
print("current_redshift:", ds.current_redshift)
print("domain_left_edge:", ds.domain_left_edge)
print("domain_right_edge:", ds.domain_right_edge)
print("particle types:", ds.particle_types)
print("fluid types:", ds.fluid_types)

yt : [INFO     ] 2026-02-10 12:33:55,172 Calculating time from 1.000e+00 to be 4.356e+17 seconds
yt : [INFO     ] 2026-02-10 12:33:55,240 Parameters: current_time              = 4.355810528213311e+17 s
yt : [INFO     ] 2026-02-10 12:33:55,241 Parameters: domain_dimensions         = [1 1 1]
yt : [INFO     ] 2026-02-10 12:33:55,242 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-02-10 12:33:55,242 Parameters: domain_right_edge         = [35000. 35000. 35000.]
yt : [INFO     ] 2026-02-10 12:33:55,243 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-02-10 12:33:55,244 Parameters: current_redshift          = 2.220446049250313e-16
yt : [INFO     ] 2026-02-10 12:33:55,244 Parameters: omega_lambda              = 0.6911
yt : [INFO     ] 2026-02-10 12:33:55,244 Parameters: omega_matter              = 0.3089
yt : [INFO     ] 2026-02-10 12:33:55,245 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-02-10 12:33:55,245 Parameters: hubble_con

ds: <class 'yt.frontends.arepo.data_structures.ArepoHDF5Dataset'>
basename: cutout_ALLFIELDS_sphere_2p1Rvir_sub488530.hdf5
current_redshift: 2.220446049250313e-16
domain_left_edge: [0. 0. 0.] code_length
domain_right_edge: [35000. 35000. 35000.] code_length
particle types: ('io',)
fluid types: ('gas', 'deposit', 'index')


In [5]:
# Cell 3 — list all available fields (raw + derived) grouped by field type
from collections import defaultdict

ftype_map = defaultdict(list)
for f in ds.field_list:
    ftype_map[f[0]].append(f[1])

print("FIELD TYPES:", sorted(ftype_map.keys()))
for ft in sorted(ftype_map.keys()):
    print(f"\n[{ft}]  n={len(ftype_map[ft])}")
    # show first ~60 fields per type to avoid wall of text
    for name in sorted(ftype_map[ft])[:60]:
        print(" ", name)
    if len(ftype_map[ft]) > 60:
        print("  ...")

yt : [INFO     ] 2026-02-10 12:34:08,209 Allocating for 4.023e+07 particles
Loading particle index: 100%|██████████| 77/77 [00:00<00:00, 6000.55it/s]


FIELD TYPES: ['PartType0', 'PartType1', 'PartType4', 'PartType5', 'all', 'nbody']

[PartType0]  n=37
  CenterOfMass
  Coordinates
  Density
  ElectronAbundance
  EnergyDissipation
  GFM_AGNRadiation
  GFM_CoolingRate
  GFM_Metallicity
  GFM_MetalsTagged
  GFM_Metals_00
  GFM_Metals_01
  GFM_Metals_02
  GFM_Metals_03
  GFM_Metals_04
  GFM_Metals_05
  GFM_Metals_06
  GFM_Metals_07
  GFM_Metals_08
  GFM_Metals_09
  GFM_WindDMVelDisp
  GFM_WindHostHaloMass
  InternalEnergy
  InternalEnergyOld
  Machnumber
  MagneticField
  MagneticFieldDivergence
  Masses
  NeutralHydrogenAbundance
  ParticleIDs
  Potential
  StarFormationRate
  SubfindDMDensity
  SubfindDensity
  SubfindHsml
  SubfindVelDisp
  Velocities
  smoothing_length

[PartType1]  n=9
  Coordinates
  Masses
  ParticleIDs
  Potential
  SubfindDMDensity
  SubfindDensity
  SubfindHsml
  SubfindVelDisp
  Velocities

[PartType4]  n=34
  BirthPos
  BirthVel
  Coordinates
  GFM_InitialMass
  GFM_Metallicity
  GFM_MetalsTagged
  GFM_Metals_

In [6]:
# Cell 4 — explicitly check for metallicity-like fields in the cutout
candidates = [
    ("gas","metallicity"),
    ("gas","GFM_Metallicity"),
    ("gas","Metallicity"),
    ("gas","Z"),
    ("PartType0","GFM_Metallicity"),
    ("PartType0","Metallicity"),
]

def exists_field(ds, f):
    try:
        return (f in ds.field_list) or (f in ds.derived_field_list)
    except Exception:
        return f in ds.field_list

print("Metallicity candidates present?")
for f in candidates:
    print(f"{f}: {exists_field(ds,f)}")

Metallicity candidates present?
('gas', 'metallicity'): True
('gas', 'GFM_Metallicity'): False
('gas', 'Metallicity'): False
('gas', 'Z'): False
('PartType0', 'GFM_Metallicity'): True
('PartType0', 'Metallicity'): False


In [7]:
# Cell 5 — list "C, Si, O, H" ion-relevant base fields that Trident may need
# This is not exhaustive; it just sanity-checks that you have the usual ingredients.
key_like = [
    "density", "temperature", "metal", "GFM", "NeutralHydrogenAbundance",
    "ElectronAbundance", "InternalEnergy", "Masses", "Coordinates",
    "Velocities", "SmoothingLength",
]

hits = []
for f in ds.field_list:
    if any(k.lower() in f[1].lower() for k in key_like):
        hits.append(f)

print("Some ion-relevant / common fields found (sample):")
for f in sorted(hits)[:120]:
    print(" ", f)
if len(hits) > 120:
    print(" ... total hits:", len(hits))

Some ion-relevant / common fields found (sample):
  ('PartType0', 'Coordinates')
  ('PartType0', 'Density')
  ('PartType0', 'ElectronAbundance')
  ('PartType0', 'GFM_AGNRadiation')
  ('PartType0', 'GFM_CoolingRate')
  ('PartType0', 'GFM_Metallicity')
  ('PartType0', 'GFM_MetalsTagged')
  ('PartType0', 'GFM_Metals_00')
  ('PartType0', 'GFM_Metals_01')
  ('PartType0', 'GFM_Metals_02')
  ('PartType0', 'GFM_Metals_03')
  ('PartType0', 'GFM_Metals_04')
  ('PartType0', 'GFM_Metals_05')
  ('PartType0', 'GFM_Metals_06')
  ('PartType0', 'GFM_Metals_07')
  ('PartType0', 'GFM_Metals_08')
  ('PartType0', 'GFM_Metals_09')
  ('PartType0', 'GFM_WindDMVelDisp')
  ('PartType0', 'GFM_WindHostHaloMass')
  ('PartType0', 'InternalEnergy')
  ('PartType0', 'InternalEnergyOld')
  ('PartType0', 'Masses')
  ('PartType0', 'NeutralHydrogenAbundance')
  ('PartType0', 'SubfindDMDensity')
  ('PartType0', 'SubfindDensity')
  ('PartType0', 'Velocities')
  ('PartType1', 'Coordinates')
  ('PartType1', 'Masses')
  ('Part

In [8]:
# Cell 6 — check Trident line database: which of your requested "lines" are recognized
requested = [
    "Si II 1206",   # likely wrong
    "C II 1334",    # Trident often uses 1335 subset name
    "C II 1302",    # likely wrong (O I 1302)
    "Si IV 1393",   # Trident often uses 1394 subset name
    "Si II 1260",
    "Si II 1190",
    "Si II 1193",
]

# Trident exposes its line database via its "LineDatabase"
ldb = trident.LineDatabase()

def has_subset(name):
    try:
        lines = ldb.parse_subset(name)
        return len(lines) > 0
    except Exception:
        return False

print("Subset recognized by Trident?")
for s in requested:
    ok = has_subset(s)
    print(f"{s:15s} -> {ok}")

yt : [INFO     ] 2026-02-10 12:34:38,504 No lines found in subset 'Si II 1206'.
yt : [INFO     ] 2026-02-10 12:34:38,505 No lines found in subset 'C II 1334'.
yt : [INFO     ] 2026-02-10 12:34:38,506 No lines found in subset 'C II 1302'.
yt : [INFO     ] 2026-02-10 12:34:38,506 No lines found in subset 'Si IV 1393'.
yt : [INFO     ] 2026-02-10 12:34:38,506 No lines found in subset 'Si II 1260'.
yt : [INFO     ] 2026-02-10 12:34:38,507 No lines found in subset 'Si II 1190'.
yt : [INFO     ] 2026-02-10 12:34:38,507 No lines found in subset 'Si II 1193'.


Subset recognized by Trident?
Si II 1206      -> False
C II 1334       -> False
C II 1302       -> False
Si IV 1393      -> False
Si II 1260      -> False
Si II 1190      -> False
Si II 1193      -> False


In [9]:
# Cell 7 — propose corrected line names (common Trident subset conventions)
# Adjust this list after you see which ones are recognized in Cell 6.
proposed = [
    "Si II 1260",
    "Si II 1190",
    "Si II 1193",
    "Si IV 1394",   # often 1394 not 1393
    "C II 1335",    # often 1335 not 1334
    "Si III 1206",  # 1206 is Si III
    "O I 1302",     # 1302 is O I
]
print("Proposed subsets recognized?")
for s in proposed:
    print(f"{s:15s} -> {has_subset(s)}")

yt : [INFO     ] 2026-02-10 12:34:55,606 No lines found in subset 'Si II 1260'.
yt : [INFO     ] 2026-02-10 12:34:55,607 No lines found in subset 'Si II 1190'.
yt : [INFO     ] 2026-02-10 12:34:55,607 No lines found in subset 'Si II 1193'.
yt : [INFO     ] 2026-02-10 12:34:55,607 No lines found in subset 'Si IV 1394'.
yt : [INFO     ] 2026-02-10 12:34:55,608 No lines found in subset 'C II 1335'.
yt : [INFO     ] 2026-02-10 12:34:55,608 No lines found in subset 'Si III 1206'.
yt : [INFO     ] 2026-02-10 12:34:55,608 No lines found in subset 'O I 1302'.


Proposed subsets recognized?
Si II 1260      -> False
Si II 1190      -> False
Si II 1193      -> False
Si IV 1394      -> False
C II 1335       -> False
Si III 1206     -> False
O I 1302        -> False


In [10]:
# Cell 8 — minimal ray build in-memory and inspect what fields are in the resulting ray dataset
# This is the exact failure mode you hit: metallicity exists on ds but not on ray dataset.
# Pick a short ray inside the domain; use two points in code_length unit box coordinates.
# IMPORTANT: Use a very short segment to keep it fast.

# crude: choose two points near the center of the domain in "code_length"
# (domain here is [0,35000] code_length typical for TNG)
p0 = ds.arr([17500.0, 17500.0, 17500.0], "code_length")
p1 = ds.arr([17510.0, 17510.0, 17510.0], "code_length")

ray = trident.make_simple_ray(ds, start_position=p0, end_position=p1)
ray_ds = getattr(ray, "ds", ray)

print("ray:", type(ray))
print("ray_ds:", type(ray_ds))
print("ray_ds field_list length:", len(ray_ds.field_list))

# list field types and a few fields
from collections import defaultdict
ft_map = defaultdict(list)
for f in ray_ds.field_list:
    ft_map[f[0]].append(f[1])
print("ray_ds FIELD TYPES:", sorted(ft_map.keys()))
for ft in sorted(ft_map.keys()):
    print(f"\n[{ft}] n={len(ft_map[ft])}")
    for name in sorted(ft_map[ft])[:40]:
        print(" ", name)
    if len(ft_map[ft]) > 40:
        print("  ...")

yt : [INFO     ] 2026-02-10 12:35:03,937 Getting segment at z = 2.220446049250313e-16: [0.5 0.5 0.5] unitary to [0.50028571 0.50028571 0.50028571] unitary.
yt : [INFO     ] 2026-02-10 12:35:03,939 Getting subsegment: [0.5 0.5 0.5] unitary to [0.50028571 0.50028571 0.50028571] unitary.


RuntimeError: No zones along ray with nonzero ('gas', 'temperature'). Modify your ray trajectory.